In [4]:
import pandas as pd 

df = pd.read_csv("../data/processed/costumer_churn_clean.csv")

print("Dimensions :", df.shape)

df.head()


Dimensions : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
print(df["Churn"].value_counts())

print(df["Churn"].value_counts(normalize=True)*100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [6]:
df["Churn"] = df["Churn"].map({
    "No" : 0 ,
    "Yes" : 1 
})

print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
X = df.drop(columns=["Churn", "customerID"])

y = df["Churn"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (7043, 19)
Dimensions de y : (7043,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (5634, 19)
X_test : (1409, 19)
y_train : (5634,)
y_test : (1409,)


In [9]:
print(X_train.dtypes)

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "gender", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]


numeric_features = [
    "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"
]


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ]
)

In [11]:

X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)


print("X_train préparé :", X_train_prepared.shape)
print("X_test préparé :", X_test_prepared.shape)

X_train préparé : (5634, 45)
X_test préparé : (1409, 45)


In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_prepared, y_train)

print("Modèle entraîné avec succès !")

Modèle entraîné avec succès !


In [13]:
y_pred = model.predict(X_test_prepared)

print("Prédictions :", y_pred[:10])

print("Valeurs réelles :", y_test.iloc[:10].values)

Prédictions : [0 1 0 0 0 1 0 0 0 0]
Valeurs réelles : [0 0 0 0 0 0 0 0 0 1]


In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :", round(accuracy * 100, 2), "%")


print(classification_report(
    y_test,
    y_pred,
    target_names=["Client reste", "Client résilie"]
))


print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Accuracy : 80.55 %
                precision    recall  f1-score   support

  Client reste       0.85      0.89      0.87      1035
Client résilie       0.66      0.56      0.60       374

      accuracy                           0.81      1409
     macro avg       0.75      0.73      0.74      1409
  weighted avg       0.80      0.81      0.80      1409

Matrice de confusion :
[[926 109]
 [165 209]]


In [15]:

model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_balanced.fit(X_train_prepared, y_train)

y_pred_balanced = model_balanced.predict(X_test_prepared)

print(classification_report(
    y_test,
    y_pred_balanced,
    target_names=["Client reste", "Client résilie"]
))

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_balanced))

                precision    recall  f1-score   support

  Client reste       0.90      0.72      0.80      1035
Client résilie       0.50      0.78      0.61       374

      accuracy                           0.74      1409
     macro avg       0.70      0.75      0.71      1409
  weighted avg       0.80      0.74      0.75      1409

Matrice de confusion :
[[747 288]
 [ 81 293]]


In [16]:
from sklearn.metrics import roc_auc_score

proba_initial = model.predict_proba(X_test_prepared)[:, 1]
proba_balanced = model_balanced.predict_proba(X_test_prepared)[:, 1]

auc_initial = roc_auc_score(y_test, proba_initial)
auc_balanced = roc_auc_score(y_test, proba_balanced)

print("ROC-AUC modèle initial :", round(auc_initial, 3))
print("ROC-AUC modèle équilibré :", round(auc_balanced, 3))

ROC-AUC modèle initial : 0.842
ROC-AUC modèle équilibré : 0.842


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score

for seuil in [0.3, 0.4, 0.5]:

    
    y_pred_seuil = (proba_initial >= seuil).astype(int)

    
    precision = precision_score(y_test, y_pred_seuil)
    recall = recall_score(y_test, y_pred_seuil)
    f1 = f1_score(y_test, y_pred_seuil)

    print(f"\nSeuil : {seuil}")
    print(f"Precision : {precision:.2f}")
    print(f"Recall : {recall:.2f}")
    print(f"F1-score : {f1:.2f}")


Seuil : 0.3
Precision : 0.52
Recall : 0.75
F1-score : 0.62

Seuil : 0.4
Precision : 0.57
Recall : 0.67
F1-score : 0.61

Seuil : 0.5
Precision : 0.66
Recall : 0.56
F1-score : 0.60


In [18]:
from sklearn.metrics import confusion_matrix


y_pred_03 = (proba_initial >= 0.3).astype(int)

cm_03 = confusion_matrix(y_test, y_pred_03)

print(cm_03)

[[774 261]
 [ 92 282]]


In [19]:
from sklearn.model_selection import train_test_split

X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)


print("Entraînement :", X_train_inner.shape)
print("Validation :", X_val.shape)

Entraînement : (4507, 19)
Validation : (1127, 19)


In [20]:

X_train_inner_prepared = preprocessor.fit_transform(X_train_inner)


X_val_prepared = preprocessor.transform(X_val)


print("Entraînement préparé :", X_train_inner_prepared.shape)
print("Validation préparée :", X_val_prepared.shape)

Entraînement préparé : (4507, 45)
Validation préparée : (1127, 45)


In [21]:

X_train_inner_prepared = preprocessor.fit_transform(X_train_inner)

X_val_prepared = preprocessor.transform(X_val)


print("Entraînement préparé :", X_train_inner_prepared.shape)
print("Validation préparée :", X_val_prepared.shape)

Entraînement préparé : (4507, 45)
Validation préparée : (1127, 45)


In [22]:
from sklearn.linear_model import LogisticRegression


model_validation = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)


model_validation.fit(
    X_train_inner_prepared,
    y_train_inner
)


proba_val = model_validation.predict_proba(X_val_prepared)[:, 1]

print("Modèle entraîné et probabilités calculées !")
print("Nombre de probabilités :", len(proba_val))

Modèle entraîné et probabilités calculées !
Nombre de probabilités : 1127


In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score

for seuil in [0.3, 0.4, 0.5]:

    
    y_pred_val = (proba_val >= seuil).astype(int)

    
    precision = precision_score(y_val, y_pred_val)
    recall = recall_score(y_val, y_pred_val)
    f1 = f1_score(y_val, y_pred_val)

    print(f"\nSeuil : {seuil}")
    print(f"Precision : {precision:.2f}")
    print(f"Recall : {recall:.2f}")
    print(f"F1-score : {f1:.2f}")


Seuil : 0.3
Precision : 0.42
Recall : 0.92
F1-score : 0.58

Seuil : 0.4
Precision : 0.46
Recall : 0.87
F1-score : 0.60

Seuil : 0.5
Precision : 0.52
Recall : 0.80
F1-score : 0.63


In [24]:

X_train_prepared_final = preprocessor.fit_transform(X_train)


X_test_prepared_final = preprocessor.transform(X_test)


from sklearn.linear_model import LogisticRegression

model_final = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)


model_final.fit(X_train_prepared_final, y_train)

print("Modèle final entraîné !")
print("Données d'entraînement :", X_train_prepared_final.shape)
print("Données de test :", X_test_prepared_final.shape)

Modèle final entraîné !
Données d'entraînement : (5634, 45)
Données de test : (1409, 45)


In [25]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


proba_test = model_final.predict_proba(X_test_prepared_final)[:, 1]


seuil_final = 0.4


y_pred_final = (proba_test >= seuil_final).astype(int)

print("Precision :", precision_score(y_test, y_pred_final))
print("Recall :", recall_score(y_test, y_pred_final))
print("F1-score :", f1_score(y_test, y_pred_final))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_final))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["Client reste", "Client résilie"]
))

Precision : 0.46618705035971225
Recall : 0.8663101604278075
F1-score : 0.6061739943872778

Matrice de confusion :
[[664 371]
 [ 50 324]]

Rapport de classification :
                precision    recall  f1-score   support

  Client reste       0.93      0.64      0.76      1035
Client résilie       0.47      0.87      0.61       374

      accuracy                           0.70      1409
     macro avg       0.70      0.75      0.68      1409
  weighted avg       0.81      0.70      0.72      1409



In [26]:
from sklearn.metrics import roc_auc_score

# Calculer le ROC-AUC sur le jeu de test
auc_final = roc_auc_score(y_test, proba_test)

print("ROC-AUC :", round(auc_final, 3))

ROC-AUC : 0.842


In [27]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(
    {
        "model": model_final,
        "preprocessor": preprocessor,
        "threshold": 0.4
    },
    "models/churn_model.pkl"
)

print("Modèle sauvegardé avec succès !")

Modèle sauvegardé avec succès !


In [28]:
import joblib


artefacts = joblib.load("models/churn_model.pkl")

model_charge = artefacts["model"]
preprocessor_charge = artefacts["preprocessor"]
seuil_charge = artefacts["threshold"]

print("Modèle chargé avec succès !")
print("Seuil utilisé :", seuil_charge)

Modèle chargé avec succès !
Seuil utilisé : 0.4


In [29]:

client = X_test.iloc[[0]]


client_prepare = preprocessor_charge.transform(client)


proba_churn = model_charge.predict_proba(client_prepare)[0, 1]


prediction = int(proba_churn >= seuil_charge)

print("Probabilité de résiliation :", round(proba_churn * 100, 2), "%")

if prediction == 1:
    print("Prédiction : risque de résiliation")
else:
    print("Prédiction : client susceptible de rester")

Probabilité de résiliation : 11.77 %
Prédiction : client susceptible de rester


In [30]:
# Récupérer la véritable valeur de churn du client
churn_reel = y_test.iloc[0]

print("Prédiction du modèle :", prediction)
print("Valeur réelle :", churn_reel)

if prediction == churn_reel:
    print("Résultat : prédiction correcte")
else:
    print("Résultat : prédiction incorrecte")

Prédiction du modèle : 0
Valeur réelle : 0
Résultat : prédiction correcte


In [31]:
# Prédire le churn pour tous les clients du test
y_pred = model_final.predict(X_test_prepared_final)

print("Prédictions terminées !")
print("Nombre de clients testés :", len(y_pred))

Prédictions terminées !
Nombre de clients testés : 1409


In [32]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy du modèle :", round(accuracy * 100, 2), "%")

Accuracy du modèle : 73.81 %


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Probabilités de résiliation
proba_test = model_final.predict_proba(X_test_prepared_final)[:, 1]

# Seuil retenu
seuil_final = 0.4

# Prédictions avec le seuil de 0,4
y_pred_final = (proba_test >= seuil_final).astype(int)

# Évaluation du modèle final
print("Seuil utilisé :", seuil_final)

print("Accuracy :", round(accuracy_score(y_test, y_pred_final) * 100, 2), "%")

print("Precision :", round(precision_score(y_test, y_pred_final), 2))
print("Recall :", round(recall_score(y_test, y_pred_final), 2))
print("F1-score :", round(f1_score(y_test, y_pred_final), 2))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_final))

Seuil utilisé : 0.4
Accuracy : 70.12 %
Precision : 0.47
Recall : 0.87
F1-score : 0.61

Matrice de confusion :
[[664 371]
 [ 50 324]]


In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

proba_test = model_final.predict_proba(X_test_prepared_final)[:, 1]

seuil_final = 0.4

y_pred_final = (proba_test >= seuil_final).astype(int)

print("Seuil utilisé :", seuil_final)

print("Accuracy :", round(accuracy_score(y_test, y_pred_final) * 100, 2), "%")

print("Precision :", round(precision_score(y_test, y_pred_final), 2))
print("Recall :", round(recall_score(y_test, y_pred_final), 2))
print("F1-score :", round(f1_score(y_test, y_pred_final), 2))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_final))

Seuil utilisé : 0.4
Accuracy : 70.12 %
Precision : 0.47
Recall : 0.87
F1-score : 0.61

Matrice de confusion :
[[664 371]
 [ 50 324]]
